# Signal Processing and Feature Extraction
---
### Dataset: Ceará State University (UECE)

NOTE: Some functions of this notebook will not work because they rely on .wav audio data or are unavailable due to being designed for parallel processing.

In [ ]:
import sys
from pathlib import Path

current_path = Path.cwd()
PROJECT_ROOT = None

for p in [current_path, current_path.parent, current_path.parent.parent]:
    if (p / "rainfall_acoustic_classification").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Project root added to sys.path: {PROJECT_ROOT}")
else:
    print("ERROR: Could not find the project root.")

## Parallel Pipeline

In [2]:
%%writefile UECE_pipeline.py
import sys
import gc
from pathlib import Path
from typing import Dict, Any, List

# ====================================================================
# 1. PATH INJECTION FOR WORKERS
# Child processes must discover the project root independently.
# ====================================================================
current_path = Path.cwd()
for p in [current_path, current_path.parent, current_path.parent.parent]:
    if (p / "rainfall_acoustic_classification").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

# ====================================================================
# 2. LIBRARY IMPORTS AND CONFIGURATION
# ====================================================================
from rainfall_acoustic_classification.core import load_audio_sample
from rainfall_acoustic_classification.processing import (
    AudioAugmenter, AudioSegmenter, AcousticMetrics
)

augmenter = AudioAugmenter(sample_rate=24000, noise_prob=0.2, cutout_prob=0.3, random_state=42)
segmenter = AudioSegmenter(sample_rate=24000,segment_duration=10.0, overlap=0.5)
metrics_ext = AcousticMetrics(sample_rate=24000, fft_window_size=1024, metric_groups=["alpha", "temporal", "spectral", "mfcc", "wavelet"])

def audio_processing_worker(row_dict: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Worker function to execute signal processing and feature extraction 
    for a single audio file.

    This function loads the audio, applies stochastic augmentation (if flagged),
    segments the signal, and extracts acoustic metrics for each segment.

    Parameters
    ----------
    row_dict : dict
        A dictionary representing a single row from the metadata DataFrame. Must contain
        at least the 'file_path', 'split', and 'should_augment' keys.

    Returns
    -------
    list of dict
        A list of dictionaries, where each dictionary contains the extracted features
        and metadata for a specific segment of the original audio file. Returns an
        empty list if the audio loading fails.
    """
    sample = load_audio_sample(row_dict.get('file_path'), sample_rate=24000)
    if not sample: 
        return []
    
    y = sample.audio_data
    aug_log = "raw"
    
    # Apply stochastic augmentation strictly if the flag is True
    if row_dict.get('split') == 'train' and row_dict.get('should_augment', False):
        y, aug_log = augmenter.process(y)
        
    results = []
    chunks = segmenter.process(y)
    
    for idx, (chunk_array, offset) in enumerate(chunks):
        features = metrics_ext.calculate(chunk_array)
        
        segment_data = row_dict.copy()
        segment_data.update({
            'segment_idx': idx, 
            'offset_sec': offset, 
            'aug_params': aug_log
        })
        segment_data.update(features)
        results.append(segment_data)
        
    # =======================================================
    # AGGRESSIVE GARBAGE COLLECTION
    # =======================================================
    del y
    del chunks
    del sample
    gc.collect() 
    
    return results

Writing UECE_pipeline.py


## Processing

In [3]:
import pandas as pd
from rainfall_acoustic_classification.utils import parallel_pipe
from UECE_pipeline import audio_processing_worker
from rainfall_acoustic_classification.utils import prepare_balanced_training_set

split_file_path = PROJECT_ROOT / "data" / "splits" / "UECE"
metrics_file_path = PROJECT_ROOT / "data" / "processed" / "UECE"

## Train

In [4]:
train_file_path = split_file_path / "UECE_train.csv"

# Load and tag the split
df_train_meta = pd.read_csv(train_file_path)
df_train_meta['split'] = 'train'

# Define the rain classes according to the methodology
RAIN_CLASSES = ['light', 'moderate', 'heavy', 'violent']

# Apply the balancing function
df_train_final = prepare_balanced_training_set(
    df_train=df_train_meta, 
    rain_classes=RAIN_CLASSES, 
    random_state=42
)

print(f"\n Total tasks dispatched to workers: {len(df_train_final)} files.")


📊 Intra-Class Balancing Strategy:
   -> Majority Class: 'moderate' with N_max = 252 samples.
   -> [light] Originals: 239 | Augmented generated: +13 | Final Total: 252
   -> [moderate] Originals: 252 | Already at N_max. No augmentation applied.
   -> [heavy] Originals: 192 | Augmented generated: +60 | Final Total: 252
   -> [violent] Originals: 32 | Augmented generated: +220 | Final Total: 252

 Total tasks dispatched to workers: 1723 files.


In [5]:
# Execute the parallel pipeline
df_train_final_features = df_train_final.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extracting UECE Features"
)

Extracting UECE Features:   0%|          | 0/1723 [00:00<?, ?file/s]

In [6]:
train_metrics_file_path = metrics_file_path / "UECE_train_metrics.csv"
df_train_final_features.to_csv(train_metrics_file_path, index=False)

## Validation

In [7]:
val_file_path = split_file_path / "UECE_val.csv"

df_val_meta = pd.read_csv(val_file_path)
df_val_meta['split'] = 'val'

df_val_final_features = df_val_meta.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extraindo Features UECE"
)

Extraindo Features UECE:   0%|          | 0/179 [00:00<?, ?file/s]

In [8]:
val_metrics_file_path = metrics_file_path / "UECE_val_metrics.csv"
df_val_final_features.to_csv(val_metrics_file_path, index=False)

## Test

In [9]:
test_file_path = split_file_path / "UECE_test.csv"

df_test_meta = pd.read_csv(test_file_path)
df_test_meta['split'] = 'test'

df_test_final_features = df_test_meta.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extraindo Features UECE"
)

Extraindo Features UECE:   0%|          | 0/179 [00:00<?, ?file/s]

In [10]:
test_metrics_file_path = metrics_file_path / "UECE_test_metrics.csv"
df_test_final_features.to_csv(test_metrics_file_path, index=False)

In [11]:
count_train = df_train_final_features['category'].value_counts()
print('TRAIN:')
print(count_train)
print("Total Samples    ", len(df_train_final_features['category']), '\n')

count_val = df_val_final_features['category'].value_counts()
print('VALIDATION:')
print(count_val)
print("Total Samples    ", len(df_val_final_features['category']), '\n')

count_test = df_test_final_features['category'].value_counts()
print('TEST')
print(count_test)
print("Total Samples    ", len(df_test_final_features['category']), '\n')

total = len(df_test_final_features['category']) + len(df_val_final_features['category']) + len(df_train_final_features['category'])

print("PERCENTAGES:")
print("Train         ", round((len(df_train_final_features['category'])/total), 2))
print("Validation    ", round((len(df_val_final_features['category'])/total), 2))
print("Test          ", round((len(df_test_final_features['category'])/total), 2))

TRAIN:
category
no-rain     7865
moderate    2772
heavy       2772
violent     2772
light       2772
Name: count, dtype: int64
Total Samples     18953 

VALIDATION:
category
no-rain     990
moderate    341
light       330
heavy       264
violent      44
Name: count, dtype: int64
Total Samples     1969 

TEST
category
no-rain     979
moderate    352
light       330
heavy       264
violent      44
Name: count, dtype: int64
Total Samples     1969 

PERCENTAGES:
Train          0.83
Validation     0.09
Test           0.09
